# Random Forest Effectiveness Classification Training

This notebook trains a Random Forest classifier using the subsidy dataset.

**Target:** `Effectiveness Label`

- `0` = Not Effective
- `1` = Moderately Effective
- `2` = Effective

The notebook evaluates:
- Accuracy
- Precision
- Recall
- F1-Score
- Specificity
- MAE
- MSE
- RMSE
- R²

The MAE, MSE, RMSE, and R² are used as **regression-style evaluation metrics for the ordinal effectiveness labels**. The primary model remains a classification model.

In [ ]:
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    cross_val_score,
    learning_curve,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

random.seed(42)
np.random.seed(42)

In [ ]:
# File and target settings

TRAIN_FILE = "subsidy_dataset2_initial (2).xlsx"
TARGET = "Effectiveness Label"

LABEL_NAMES = {
    0: "Not Effective",
    1: "Moderately Effective",
    2: "Effective"
}

In [ ]:
# Load dataset

train_df = pd.read_excel(TRAIN_FILE)

print("Dataset shape:", train_df.shape)
display(train_df.head())

In [ ]:
# Dataset information

print("Column information:")
train_df.info()

print("\nMissing values:")
display(train_df.isnull().sum().to_frame("Missing Values"))

print("\nEffectiveness Label distribution:")
display(train_df[TARGET].value_counts().sort_index())

In [ ]:
# Features and target

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]

categorical_cols = [
    "Subsidy Received"
]

numerical_cols = [
    "Farm Size (ha)",
    "Average Yield (bags/ha)",
    "Crop Yield (bags/ha)",
    "Average Selling Price (₱/kg)",
    "Feedback Score"
]

print("Features:")
display(X.head())

print("\nTarget:")
display(y.head())

In [ ]:
# Preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            "passthrough",
            numerical_cols
        )
    ]
)

In [ ]:
# Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Random Forest pipeline

pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])

# Anti-overfitting search space
params = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [5, 8, 10, 12, 15],
    "classifier__min_samples_split": [5, 10, 15, 20],
    "classifier__min_samples_leaf": [2, 4, 6, 8],
    "classifier__max_features": ["sqrt", "log2"],
    "classifier__bootstrap": [True]
}

In [ ]:
# Hyperparameter tuning and training

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=params,
    n_iter=40,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

model = search.best_estimator_

print("\nBest parameters:")
print(search.best_params_)

In [ ]:
# Predictions and accuracy

y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_pred)
overfit_gap = train_acc - test_acc

print(f"Training Accuracy : {train_acc * 100:.2f}%")
print(f"Testing Accuracy  : {test_acc * 100:.2f}%")
print(f"Train-Test Gap    : {overfit_gap * 100:.2f} percentage points")

In [ ]:
# Classification report

report = classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    target_names=[
        "Not Effective",
        "Moderately Effective",
        "Effective"
    ],
    digits=4,
    output_dict=True
)

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    target_names=[
        "Not Effective",
        "Moderately Effective",
        "Effective"
    ],
    digits=4
))

In [ ]:
# Confusion matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1, 2]
)

print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Not Effective",
        "Moderately Effective",
        "Effective"
    ]
)

disp.plot()
plt.title("Random Forest - Effectiveness Label")
plt.tight_layout()
plt.savefig("confusion_matrix_test.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Specificity for each effectiveness class

specificity_values = []

for i in range(len(cm)):
    true_negative = (
        cm.sum()
        - cm[i, :].sum()
        - cm[:, i].sum()
        + cm[i, i]
    )

    false_positive = cm[:, i].sum() - cm[i, i]

    specificity = (
        true_negative / (true_negative + false_positive)
        if (true_negative + false_positive) > 0
        else 0
    )

    specificity_values.append(specificity)

    print(
        f"{LABEL_NAMES[i]} Specificity: "
        f"{specificity:.4f}"
    )

weighted_specificity = np.mean(specificity_values)

print(f"\nAverage Specificity: {weighted_specificity:.4f}")

In [ ]:
# Regression-style evaluation of Effectiveness Label

# 0 = Not Effective
# 1 = Moderately Effective
# 2 = Effective

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Regression-Style Evaluation")
print("===========================")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print(f"\nMAE: predictions differ from actual labels by")
print(f"approximately {mae:.4f} label levels on average.")

print(f"R²: approximately {r2 * 100:.2f}% of the variation")
print("in the numerical effectiveness labels is explained by the predictions.")

In [ ]:
# 5-fold cross-validation

cv = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

cv_mean = cv.mean()
cv_std = cv.std()

print("CV Scores:", cv)
print(f"CV Mean Accuracy: {cv_mean * 100:.2f}%")
print(f"CV Std. Deviation: {cv_std * 100:.2f}%")

In [ ]:
# Learning curve

sizes, train_scores, valid_scores = learning_curve(
    model,
    X,
    y,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring="accuracy",
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(sizes, train_mean, marker="o", label="Training")
plt.plot(sizes, valid_mean, marker="o", label="Validation")
plt.xlabel("Training Samples")
plt.ylabel("Accuracy")
plt.title("Random Forest Learning Curve")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("learning_curve.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Feature importance

feature_names = (
    model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

feature_importance = (
    model
    .named_steps["classifier"]
    .feature_importances_
)

fi = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importance
}).sort_values(
    by="Importance",
    ascending=False
)

display(fi.head(10))

fi.to_excel("feature_importance.xlsx", index=False)

print("Feature importance saved to feature_importance.xlsx")

In [ ]:
# Export all evaluation metrics

metrics = pd.DataFrame({
    "Metric": [
        "Train Accuracy",
        "Test Accuracy",
        "Train-Test Gap",
        "Cross Validation Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "Specificity",
        "MAE",
        "MSE",
        "RMSE",
        "R2"
    ],
    "Value": [
        train_acc,
        test_acc,
        overfit_gap,
        cv_mean,
        report["weighted avg"]["precision"],
        report["weighted avg"]["recall"],
        report["weighted avg"]["f1-score"],
        weighted_specificity,
        mae,
        mse,
        rmse,
        r2
    ]
})

display(metrics)

metrics.to_excel(
    "training_metrics.xlsx",
    index=False
)

print("Metrics saved to training_metrics.xlsx")

In [ ]:
# Save trained model

joblib.dump(
    model,
    "random_forest_subsidy.pkl"
)

print("Model saved as random_forest_subsidy.pkl")

In [ ]:
# Final summary

print("=" * 50)
print("FINAL MODEL SUMMARY")
print("=" * 50)

print(f"Training Accuracy : {train_acc * 100:.2f}%")
print(f"Testing Accuracy  : {test_acc * 100:.2f}%")
print(f"CV Accuracy       : {cv_mean * 100:.2f}%")
print(f"Specificity       : {weighted_specificity:.4f}")

print("\nRegression-Style Evaluation")
print(f"MAE               : {mae:.4f}")
print(f"MSE               : {mse:.4f}")
print(f"RMSE              : {rmse:.4f}")
print(f"R²                : {r2:.4f}")

print("\nGenerated Files:")
print("- random_forest_subsidy.pkl")
print("- confusion_matrix_test.png")
print("- learning_curve.png")
print("- feature_importance.xlsx")
print("- training_metrics.xlsx")